<a href="https://colab.research.google.com/github/TesterSim2/Focused-IDE/blob/main/Slop_to_Secure_(Vibe_Code_Reworking).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U vllm transformers accelerate numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.

In [ ]:
# ==========================================
# CELL 1: DEPENDENCIES & INFERENCE ENGINE
# ==========================================
!pip install -qU "vllm" "openai" "tenacity"
# Pinning Tree-sitter to guarantee deterministic API behavior
!pip install -q "tree-sitter==0.21.3" "tree-sitter-python==0.21.0"

import subprocess
import time
import urllib.request
import sys

# We use an 8B model to easily fit within the 80GB A100 alongside our Orchestrator
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
PORT = 8000
LOG_FILE = "vllm_server.log"

print(f"🚀 Initiating vLLM Server targeting: {MODEL_ID}")
print("⏳ Loading weights into A100 VRAM (This takes ~2-4 minutes)...")

command =[
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--served-model-name", "coder-8b",
    "--port", str(PORT),
    "--gpu-memory-utilization", "0.70", # Leave 30% VRAM for Orchestrator/Tree-sitter
    "--max-model-len", "16384",
    "--dtype", "bfloat16"
]

server_log = open(LOG_FILE, "w")
vllm_process = subprocess.Popen(command, stdout=server_log, stderr=subprocess.STDOUT)

# Wait for health check
start_time = time.time()
while True:
    if vllm_process.poll() is not None:
        print("\n❌ FATAL ERROR: vLLM process crashed! Check vllm_server.log")
        sys.exit(1)
    try:
        req = urllib.request.Request(f"http://localhost:{PORT}/v1/models", method="GET")
        with urllib.request.urlopen(req) as response:
            if response.getcode() == 200:
                print("\n✅ vLLM Server is HEALTHY and listening on port 8000!")
                break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(5)

    if time.time() - start_time > 600:
        print("\n❌ TIMEOUT: Server took longer than 10 minutes.")
        sys.exit(1)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.6/130.6 kB 16.4 MB/s eta 0:00:00
🚀 Initiating vLLM Server targeting: Qwen/Qwen2.5-Coder-7B-Instruct
⏳ Loading weights into A100 VRAM (This takes ~2-4 minutes)...
................................
✅ vLLM Server is HEALTHY and listening on port 8000!


In [ ]:
# ==========================================
# CELL 2: THE SANDBOX WORKER (RPC SCRIPT)
# ==========================================
%%writefile worker.py
import sys
import json
import importlib
import inspect
from unittest.mock import MagicMock, patch
from typing import Callable, List, Any, Dict
import io
import contextlib

class DeterministicProxy(MagicMock):
    """The Hollow-Core nested proxy to prevent Data Starvation."""
    def __init__(self, *args, _name_=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._name_ = _name_ or "Proxy"

    def __getattr__(self, name):
        if name.startswith("_"):
            return super().__getattr__(name)
        return DeterministicProxy(_name_=f"{self._name_}.{name}")

    def __call__(self, *args, **kwargs):
        super().__call__(*args, **kwargs)
        return DeterministicProxy(_name_=f"{self._name_}()")

    def __str__(self):
        return f"<DeterministicProxy: {self._name_}>"

def synthesize_args(func: Callable) -> tuple:
    """Dynamically satisfies arbitrary arity for enterprise functions."""
    sig = inspect.signature(func)
    args, kwargs =[], {}
    for name, param in sig.parameters.items():
        if param.default == inspect.Parameter.empty:
            if param.kind in (inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD):
                args.append(DeterministicProxy(_name_=f"arg_{name}"))
            elif param.kind == inspect.Parameter.KEYWORD_ONLY:
                kwargs[name] = DeterministicProxy(_name_=f"kwarg_{name}")
    return args, kwargs

class ExecutionTracer:
    """Isolates side-effects and generates the Golden Master Call Graph."""
    def __init__(self, target_module, external_imports: List[str]):
        self.target_module = target_module
        self.external_imports = external_imports
        self.patches = []
        self.call_graph: List[str] =[]

    def __enter__(self):
        for imp in self.external_imports:
            target_path = f"{self.target_module.__name__}.{imp}"
            try:
                p = patch(target_path, new_callable=DeterministicProxy, _name_=imp)
                mock_obj = p.start()
                self.patches.append((p, mock_obj))
            except AttributeError:
                pass
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        for p, mock_obj in self.patches:
            for call in mock_obj.mock_calls:
                self.call_graph.append(f"{mock_obj._name_}.{call}")
            p.stop()

def run_trace(module_path: str, func_name: str, code_override: str, external_imports: List[str]) -> dict:
    try:
        target_module = importlib.import_module(module_path)

        if code_override:
            # Overwrite the function in the module's namespace
            exec(code_override, target_module.__dict__)

        target_func = getattr(target_module, func_name)
        args, kwargs = synthesize_args(target_func)
        tracer = ExecutionTracer(target_module, external_imports)

        # Prevent slop from polluting STDOUT (which breaks our JSON RPC)
        dummy_stdout = io.StringIO()
        dummy_stderr = io.StringIO()

        with tracer:
            with contextlib.redirect_stdout(dummy_stdout), contextlib.redirect_stderr(dummy_stderr):
                try:
                    target_func(*args, **kwargs)
                except Exception:
                    pass

        return {"status": "success", "call_graph": tracer.call_graph}

    except Exception as e:
        return {"status": "error", "message": str(e)}

if __name__ == "__main__":
    payload = json.loads(sys.stdin.read())
    result = run_trace(
        module_path=payload["module_path"],
        func_name=payload["func_name"],
        code_override=payload.get("code_override"),
        external_imports=payload.get("external_imports",[])
    )
    print(json.dumps(result))

Writing worker.py


In [ ]:
# ==========================================
# CELL 3: ENTERPRISE CORE ARCHITECTURE (PATCHED V2)
# ==========================================
import os
import sys
import json
import re
import subprocess
import tree_sitter
import tree_sitter_python as tspython
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from pathlib import Path
from openai import OpenAI

# ---------------------------------------------------------
# 1. DATA MODELS & LEDGER
# ---------------------------------------------------------
@dataclass
class KnowledgeComponent:
    identifier: str
    start_byte: int
    end_byte: int
    cyclomatic_complexity: int
    raw_bytes: bytes
    file_path: str = ""

@dataclass
class StateLedger:
    pending_tasks: List[KnowledgeComponent] = field(default_factory=list)
    completed_tasks: List[KnowledgeComponent] = field(default_factory=list)
    failed_tasks: List[dict] = field(default_factory=list)

    def pop_task(self) -> Optional[KnowledgeComponent]:
        return self.pending_tasks.pop(0) if self.pending_tasks else None

# ---------------------------------------------------------
# 2. SPATIAL COMPRESSION (Tree-sitter)
# ---------------------------------------------------------
class SpatialCompressionLayer:
    def __init__(self):
        self.ts_lang = tree_sitter.Language(tspython.language(), "python")
        self.parser = tree_sitter.Parser()
        # FIX: Used set_language() instead of property assignment
        self.parser.set_language(self.ts_lang)

        self.func_query = self.ts_lang.query("""
            (function_definition name: (identifier) @func_name) @func_def
        """)
        self.complexity_query = self.ts_lang.query("""[(if_statement) (for_statement) (while_statement)
             (except_clause) (with_statement) (list_comprehension)] @branch
        """)

    def ingest_file(self, file_path: str) -> List[KnowledgeComponent]:
        with open(file_path, "rb") as f:
            source_bytes = f.read()

        tree = self.parser.parse(source_bytes)
        components =[]

        captures = self.func_query.captures(tree.root_node)

        current_def = None
        current_name = "anonymous"

        for node, capture_name in captures:
            if capture_name == "func_def":
                current_def = node
            elif capture_name == "func_name":
                current_name = node.text.decode('utf8')

            if current_def and current_name != "anonymous":
                complexity = len(self.complexity_query.captures(current_def)) + 1
                kc = KnowledgeComponent(
                    identifier=current_name,
                    start_byte=current_def.start_byte,
                    end_byte=current_def.end_byte,
                    cyclomatic_complexity=complexity,
                    raw_bytes=source_bytes[current_def.start_byte:current_def.end_byte]
                )
                components.append(kc)
                current_def = None
                current_name = "anonymous"

        return components

class ImportDependencyExtractor:
    def __init__(self):
        self.ts_lang = tree_sitter.Language(tspython.language(), "python")
        self.parser = tree_sitter.Parser()
        # FIX: Used set_language() instead of property assignment
        self.parser.set_language(self.ts_lang)

        self.stdlib_names = getattr(sys, 'stdlib_module_names', set([
            "json", "math", "os", "sys", "re", "datetime", "typing",
            "collections", "itertools", "functools", "pathlib", "hashlib"
        ]))

        self.import_query = self.ts_lang.query("""
            (import_statement name: (dotted_name) @module)
            (import_from_statement module_name: (dotted_name) @module)
            (import_from_statement module_name: (relative_import) @module)
            (import_from_statement (relative_import) @module)
        """)

    def extract_side_effect_imports(self, file_bytes: bytes) -> List[str]:
        tree = self.parser.parse(file_bytes)
        captures = self.import_query.captures(tree.root_node)

        modules_to_mock = set()
        for node, _ in captures:
            module_name = node.text.decode('utf-8')
            root_module = module_name.split('.')[0]
            if root_module and root_module not in self.stdlib_names:
                modules_to_mock.add(module_name)

        return list(modules_to_mock)

class DeterministicPatcher:
    @staticmethod
    def apply_patch(original_file_path: str, component: KnowledgeComponent, new_code_bytes: bytes) -> bool:
        try:
            with open(original_file_path, "rb") as f:
                source_bytes = f.read()

            patched_bytes = source_bytes[:component.start_byte] + new_code_bytes + source_bytes[component.end_byte:]

            parser = tree_sitter.Parser()
            ts_lang = tree_sitter.Language(tspython.language(), "python")
            # FIX: Used set_language() instead of property assignment
            parser.set_language(ts_lang)
            test_tree = parser.parse(patched_bytes)

            if test_tree.root_node.has_error:
                return False

            with open(original_file_path, "wb") as f:
                f.write(patched_bytes)
            return True

        except Exception as e:
            print(f"Patcher Error: {e}")
            return False

# ---------------------------------------------------------
# 3. RPC SANDBOX BRIDGE
# ---------------------------------------------------------
class SandboxBridge:
    def __init__(self, repo_root: str):
        self.repo_root = repo_root

    def get_call_graph(self, module_path: str, func_name: str, external_imports: List[str], candidate_code: str = None) -> List[str]:
        payload = {
            "module_path": module_path,
            "func_name": func_name,
            "external_imports": external_imports,
            "code_override": candidate_code
        }

        env = os.environ.copy()
        env["PYTHONPATH"] = self.repo_root

        cmd = [sys.executable, "worker.py"]

        process = subprocess.run(
            cmd,
            input=json.dumps(payload).encode('utf-8'),
            capture_output=True,
            timeout=10,
            env=env
        )

        try:
            response = json.loads(process.stdout.decode('utf-8'))
        except json.JSONDecodeError:
            error_output = process.stderr.decode('utf-8')
            raise RuntimeError(f"Sandbox Corrupted JSON. Stderr: {error_output}")

        if response["status"] == "error":
            raise RuntimeError(response["message"])

        return response["call_graph"]

# ---------------------------------------------------------
# 4. AMNESIA ORCHESTRATOR
# ---------------------------------------------------------
class GuidedAmnesiaOrchestrator:
    def __init__(self, patcher: DeterministicPatcher, max_retries: int = 3):
        self.llm = OpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")
        self.patcher = patcher
        self.max_retries = max_retries
        self.ledger = StateLedger()

    def _extract_code(self, raw_text: str) -> str:
        match = re.search(r"```python\n(.*?)```", raw_text, re.DOTALL | re.IGNORECASE)
        return match.group(1).strip() if match else raw_text.strip()

    def run_amnesia_loop(self, component: KnowledgeComponent, sandbox: SandboxBridge, module_path: str, external_imports: List[str]) -> bool:
        print(f"  └─ Acquiring Golden Master Trace...")
        try:
            golden_graph = sandbox.get_call_graph(module_path, component.identifier, external_imports)
        except Exception as e:
            print(f"  ❌ Dirty Import or Native Sandbox Crash: {e}. Skipping component.")
            return False

        base_prompt = (
            "You are a strict Python refactoring engine. Refactor this function to be clean, DRY, and strictly typed.\n"
            "CRITICAL CONSTRAINTS:\n"
            "1. Do NOT alter the function name or its arguments.\n"
            "2. Do NOT remove any existing side-effects (database calls, API requests).\n"
            "3. Output ONLY valid Python code inside a ```python block. NO explanations.\n\n"
            f"TARGET CODE:\n```python\n{component.raw_bytes.decode('utf-8')}\n```"
        )
        current_prompt = base_prompt

        for attempt in range(1, self.max_retries + 1):
            print(f"  └─ Amnesia Loop Attempt {attempt}/{self.max_retries}...")

            response = self.llm.chat.completions.create(
                model="coder-8b",
                messages=[{"role": "user", "content": current_prompt}],
                temperature=0.1,
                max_tokens=2048
            )
            candidate_code_str = self._extract_code(response.choices[0].message.content)

            try:
                candidate_graph = sandbox.get_call_graph(module_path, component.identifier, external_imports, candidate_code_str)
            except RuntimeError as e:
                current_prompt = f"{base_prompt}\n\n⚠️ PREVIOUS ATTEMPT CAUSED EXCEPTION: {e}\nFix the logic."
                continue
            except subprocess.TimeoutExpired:
                current_prompt = f"{base_prompt}\n\n⚠️ PREVIOUS ATTEMPT CAUSED INFINITE LOOP (Timeout).\nFix the logic."
                continue

            if golden_graph == candidate_graph:
                print(f"  ✅ Equivalence Verified. Graphs match.")
                if self.patcher.apply_patch(component.file_path, component, candidate_code_str.encode('utf-8')):
                    return True
                else:
                    current_prompt = f"{base_prompt}\n\n⚠️ SYNTAX ERROR: Your code caused a Tree-sitter parse error. Ensure valid Python."
            else:
                print(f"  ❌ Side-Effect Mismatch. Generating Constraints.")
                current_prompt = (
                    f"{base_prompt}\n\n"
                    "⚠️ PREVIOUS ATTEMPT FAILED BEHAVIORAL EQUIVALENCE ⚠️\n"
                    f"EXPECTED GRAPH: {json.dumps(golden_graph, indent=2)}\n"
                    f"ACTUAL GRAPH:   {json.dumps(candidate_graph, indent=2)}\n"
                    "Generate a new refactor perfectly matching the EXPECTED GRAPH."
                )

        return False

# ---------------------------------------------------------
# 5. OVERARCHING PIPELINE
# ---------------------------------------------------------
class EnterpriseRefactorPipeline:
    def __init__(self, repo_root: str, complexity_threshold: int = 5):
        self.repo_root = Path(repo_root).resolve()
        self.complexity_threshold = complexity_threshold

        self.mapper = SpatialCompressionLayer()
        self.patcher = DeterministicPatcher()
        self.dependency_extractor = ImportDependencyExtractor()
        self.sandbox = SandboxBridge(str(self.repo_root))
        self.orchestrator = GuidedAmnesiaOrchestrator(self.patcher)

    def scan_repository(self):
        print(f"\n🔍 Scanning Repository: {self.repo_root}")
        for file_path in self.repo_root.rglob("*.py"):
            if any(p.startswith('.') for p in file_path.parts) or "venv" in file_path.parts:
                continue
            try:
                components = self.mapper.ingest_file(str(file_path))
                for kc in components:
                    if kc.cyclomatic_complexity >= self.complexity_threshold:
                        kc.file_path = str(file_path)
                        self.orchestrator.ledger.pending_tasks.append(kc)
            except Exception as e:
                print(f"⚠️ Exception during parsing {file_path}: {e}")

        self.orchestrator.ledger.pending_tasks.sort(key=lambda x: x.cyclomatic_complexity, reverse=True)
        print(f"📊 Identified {len(self.orchestrator.ledger.pending_tasks)} high-complexity components.")

    def run(self):
        self.scan_repository()

        while task := self.orchestrator.ledger.pop_task():
            print(f"\n{'='*60}\n🚀 PROCESSING: {task.identifier} | Slop Score: {task.cyclomatic_complexity}\n{'='*60}")

            with open(task.file_path, "rb") as f:
                file_bytes = f.read()
            external_deps = self.dependency_extractor.extract_side_effect_imports(file_bytes)

            rel_path = Path(task.file_path).relative_to(self.repo_root)
            module_path = str(rel_path.with_suffix('')).replace(os.sep, '.')

            if module_path.endswith("__init__"):
                continue

            success = self.orchestrator.run_amnesia_loop(task, self.sandbox, module_path, external_deps)

            if success:
                self.orchestrator.ledger.completed_tasks.append(task)
            else:
                self.orchestrator.ledger.failed_tasks.append({"task": task.identifier, "reason": "Failed Loop"})

        self._generate_report()

    def _generate_report(self):
        print(f"\n{'='*60}\n🏁 ENTERPRISE REFACTOR PIPELINE COMPLETE")
        print(f"✅ Successfully Refactored: {len(self.orchestrator.ledger.completed_tasks)}")
        print(f"❌ Failed/Skipped: {len(self.orchestrator.ledger.failed_tasks)}")
        print(f"{'='*60}")

In [ ]:
# ==========================================
# CELL 4: SYNTHETIC TEST HARNESS & EXECUTION
# ==========================================
import os
from pathlib import Path

# ---------------------------------------------------------
# Step 1: Scaffold the Client Repository
# ---------------------------------------------------------
repo_dir = Path("/content/synthetic_slop_repo")
repo_dir.mkdir(parents=True, exist_ok=True)
(repo_dir / "__init__.py").touch()

# The Monolith: High Cyclomatic Complexity, mixing math (safe) and requests (hollowed)
monolith_code = """
import json
import math
import requests

def process_data_monolith(user_id: int):
    # This loop drives cyclomatic complexity through the roof
    results =[]
    response = requests.get(f"https://api.example.com/users/{user_id}/data")

    if response.status_code == 200:
        raw_data = json.loads(response.text)
        for item in raw_data.get("items",[]):
            if item.get("active", False):
                if item.get("value", 0) > 10:
                    calculated = math.sqrt(item["value"])
                    results.append(calculated)
                    requests.post("https://api.example.com/log", json={"val": calculated})
                else:
                    requests.post("https://api.example.com/log", json={"val": "low_value"})
            elif item.get("pending", False):
                results.append(0)
    else:
        requests.post("https://api.example.com/error", data="Failed to fetch user data")

    return results
"""
with open(repo_dir / "slop_monolith.py", "w") as f:
    f.write(monolith_code.strip())


# The Dirty Import: Side-effects at the global scope
dirty_import_code = """
import requests

# This executes purely on import - the Sandbox should crash and reject this!
# A classic legacy code anti-pattern.
CRITICAL_STATE = requests.get("https://api.example.com/init").json()

def innocent_looking_function():
    if CRITICAL_STATE.get("ready"):
        return True
    return False
"""
with open(repo_dir / "slop_dirty.py", "w") as f:
    f.write(dirty_import_code.strip())

# ---------------------------------------------------------
# Step 2: Ignite the Enterprise Pipeline
# ---------------------------------------------------------
print("\n" + "="*60)
print("🧪 INITIALIZING SYNTHETIC SLOP TEST ENVIRONMENT")
print("="*60)

# We set the complexity threshold to 4 to ensure it catches our monolith
pipeline = EnterpriseRefactorPipeline(
    repo_root=str(repo_dir),
    complexity_threshold=4
)

# Execute the pipeline!
pipeline.run()

# ---------------------------------------------------------
# Step 3: File System Verification
# ---------------------------------------------------------
print("\n" + "="*60)
print("📄 VERIFYING PHYSICAL FILE PATCH (slop_monolith.py):")
print("="*60)
with open(repo_dir / "slop_monolith.py", "r") as f:
    patched_content = f.read()
    print(patched_content)

print("\n" + "="*60)
print("🔬 END OF SYSTEM EXECUTION")
print("="*60)


🧪 INITIALIZING SYNTHETIC SLOP TEST ENVIRONMENT

🔍 Scanning Repository: /content/synthetic_slop_repo
📊 Identified 1 high-complexity components.

🚀 PROCESSING: process_data_monolith | Slop Score: 5
  └─ Acquiring Golden Master Trace...
  └─ Amnesia Loop Attempt 1/3...
  ✅ Equivalence Verified. Graphs match.

🏁 ENTERPRISE REFACTOR PIPELINE COMPLETE
✅ Successfully Refactored: 1
❌ Failed/Skipped: 0

📄 VERIFYING PHYSICAL FILE PATCH (slop_monolith.py):
import json
import math
import requests

import requests
import json
import math

def process_data_monolith(user_id: int):
    def log_value(val):
        requests.post("https://api.example.com/log", json={"val": val})

    def handle_active_item(item):
        value = item.get("value", 0)
        if value > 10:
            calculated = math.sqrt(value)
            log_value(calculated)
            return calculated
        else:
            log_value("low_value")
            return None

    def handle_pending_item():
        return 0

    re